In [1]:
import requests
import csv
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

CSV_FILE = "bbc.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.bbc.com/",
}

def load_existing_urls():
    """Load already-scraped URLs from CSV."""
    if not os.path.exists(CSV_FILE):
        return set()
    with open(CSV_FILE, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return {row["url"] for row in reader if "url" in row}

def get_article_links(base_url):
    """
    Get all unique BBC article links from a BBC section page.
    """
    try:
        resp = requests.get(base_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        links = set()
        for a in soup.find_all("a", href=True):
            href = a["href"]

            # Keep only proper article URLs
            if href.startswith("/news/articles/"):
                full_url = urljoin("https://www.bbc.com", href)
                links.add(full_url)

        return sorted(links)

    except Exception as e:
        print(f"Error fetching {base_url}: {e}")
        return []

def scrape_article_bbc(url):
    """Scrape the main BBC article body text, including bullet points."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        article_tag = soup.find("article")
        if not article_tag:
            return "", "", ""

        paragraphs = []
        # Loop through all top-level tags inside the article
        for elem in article_tag.find_all(["p", "li"], recursive=True):
            text = elem.get_text(strip=True)
            if not text:
                continue
            if text.lower().startswith(("related topics", "more on this story")):
                break
            paragraphs.append(text)

        title_tag = soup.find("h1")
        title_text = title_tag.get_text(strip=True) if title_tag else ""

        time_tag = soup.find("time")
        published_date = time_tag.get("datetime") if time_tag else ""

        return title_text, published_date, "\n".join(paragraphs)

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "", "", ""


def save_articles(articles):
    """Save scraped articles to CSV."""
    file_exists = os.path.exists(CSV_FILE)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["title", "url", "published_date", "content"])
        if not file_exists:
            writer.writeheader()
        for article in articles:
            writer.writerow(article)

if __name__ == "__main__":
    existing_urls = load_existing_urls()
    print(f"Loaded {len(existing_urls)} already-scraped URLs.")

    base_urls = [
        "https://www.bbc.com/news",
        "https://www.bbc.com/innovation"
    ]

    all_links = set()
    for url in base_urls:
        section_links = get_article_links(url)
        print(f"Found {len(section_links)} articles from {url}")
        all_links.update(section_links)
        time.sleep(5)

    print(f"Total unique articles found: {len(all_links)}")

    scraped_data = []
    for link in all_links:
        if link in existing_urls:
            print(f"Skipping already-scraped article: {link}")
            continue
        print(f"Scraping: {link}")
        time.sleep(10)  # polite delay
        title, published_date, content = scrape_article_bbc(link)
        if content:
            scraped_data.append({
                "title": title,
                "url": link,
                "published_date": published_date,
                "content": content
            })

    if scraped_data:
        save_articles(scraped_data)
        print(f"Saved {len(scraped_data)} new articles.")
    else:
        print("No new articles found.")


Loaded 35 already-scraped URLs.
Found 24 articles from https://www.bbc.com/news
Found 14 articles from https://www.bbc.com/innovation
Total unique articles found: 35
Skipping already-scraped article: https://www.bbc.com/news/articles/cwy003l0pw0o
Skipping already-scraped article: https://www.bbc.com/news/articles/cwy3ngr2n3vo
Skipping already-scraped article: https://www.bbc.com/news/articles/cy5prvgw0r1o
Skipping already-scraped article: https://www.bbc.com/news/articles/ckglnk6yxlko
Skipping already-scraped article: https://www.bbc.com/news/articles/c4gerg74y71o
Skipping already-scraped article: https://www.bbc.com/news/articles/c0e9py7e28xo
Skipping already-scraped article: https://www.bbc.com/news/articles/c62w8ewg849o
Skipping already-scraped article: https://www.bbc.com/news/articles/cgkrge6e0z4o
Skipping already-scraped article: https://www.bbc.com/news/articles/c741qken2wvo
Skipping already-scraped article: https://www.bbc.com/news/articles/ce830l92p68o
Skipping already-scraped